# GPS og lineær algebra

## Innføring

GPS bruker tiden til forskjellige satelitter til å bestemme hvor vi er på jorden.
Har vi posisjonene $(x_i, y_i, z_i)$ i kartesiske koordinater, hvor origoen er jordens sentrum, $z$-aksen peker mot nordpolen, $x$-aksen peker mot $0^\circ N, 0^\circ E$, og $y$-aksen peker mot $0^\circ N, 90^\circ E$. Avstand til satelitten er $R_i$, som vi finner ved å gange tiden det tar mellom signalen er sendt og mottatt med lyshastigheten.

Her er noen data fra virkelighet:

In [1]:
import numpy as np

c=299792458

v = np.array([[17320278.617,  -8377650.251, 18417718.933], # satellite positions
              [23903923.512,   -575354.301, 12082414.203],
              [18145496.573,    573199.862, 19274924.497],
#              [ 4943294.735, -14506778.876, 21432563.831],
#              [13257395.396,  20015827.354, 11298737.140],
              [10229840.139,  10977541.920, 21818177.084]])

R = np.array([21166927.434, # measured distance to each satellite
              21787043.623, 
              20356882.560, 
#              22039726.386,
#              22639167.771, 
              20600717.361]) 

q = np.size(v,0)

Hvordan finner vi ut hvor vi er?

## Bancroft-metoden

Det finnes en smart metode for å gjøre om GPS-ligningen til et lineært system. Først definerer vi en modifisert skalarprodukt, den Lorentz-produkten (mye brukt i relativitetstoeri):

$\langle \vec{u},\vec{v}\rangle = u_1v_1 + u_2 v_2 + u_3v_3 - u_4 v_4$

Da setter vi opp følgende matrisa og vektorer:

$
B = \begin{pmatrix} x_1 & y_1 & z_1 & R_1 \\ x_2 & y_2 & z_2 & R_2 \\ \cdots & \cdots & \cdots & \cdots
\end{pmatrix},
\qquad
\vec{a} = \frac{1}{2}
\begin{pmatrix} \langle s_1, s_1\rangle \\ \langle s_2, s_2 \rangle \\ \vdots \end{pmatrix},
\qquad
\vec{b} = 
\begin{pmatrix} 1 \\ 1 \\ \vdots \end{pmatrix},
$

Hvor $s_i = (x_i, y_i, z_i, R_i)$. Begynn ved å sett opp disse matrisene/vektorer i python under:

In [4]:
B = np.hstack((v, -R.reshape(q,1)))

vec_a = (np.linalg.norm(v,axis=1)**2 - R**2)/2

e = np.ones((q))

Ba = np.linalg.solve(B,vec_a)
Be = np.linalg.solve(B,e)

print(vec_a)
print(Ba)
print(Be)

[1.30675314e+14 1.21519027e+14 1.43353825e+14 1.38399676e+14]
[3045749.38416784  572917.37090684 5251284.74594999  661168.16945593]
[ 6.34495044e-09  1.28233065e-09  1.13703308e-08 -3.26656315e-08]


## Den kvadratiske ligningen

For å finne ut hvor vi er må vi først løse en kvadratisk ligning for $\lambda$:

$\langle B^{-1}\vec{e}, B^{-1}\vec{e} \rangle \lambda^2
+ 2\big(\langle B^{-1}\vec{a}, B^{-1}\vec{e}\rangle - 1\big) \lambda
+ \langle B^{-1}\vec{a}, B^{-1}\vec{a}\rangle = 0  $

Vi får to løsninger $\lambda_1, \lambda_2$. Bare en er riktig, men vi vet ikke hvilken ennå! Finn løsningen under:

In [3]:
def mip(u,v):
    v[3]=-v[3]
    return np.dot(u,v)

a = mip(Be,Be)
b = 2*(mip(Ba,Be)-1)
c = mip(Ba,Ba)

l1 = (-b - np.sqrt(b**2 - 4*a*c))/(2*a)
l2 = (-b + np.sqrt(b**2 - 4*a*c))/(2*a)

print(l1)
print(l2)

20239987958104.785
1501009412603702.5


## Det lineære systemet

Nå løser vi følgende system for $\vec{u}$ 

$B\vec{u} = \vec{a} + \lambda_i\vec{e}$

In [4]:
u1 = np.linalg.solve(B, vec_a + l1*e)
u2 = np.linalg.solve(B, vec_a + l2*e)

print(np.linalg.norm(u1[0:3]))
print(np.linalg.norm(u2[0:3]))

6362387.5955452295
25735919.13791787


Hvilken $i$ bruker vi, 1 eller 2? Svaret: den som er riktig! Hvordan vet det? Svaret er at bare en gir en $\vec{u}$ som gir mening. For å finne ut hvilken (og tolke resultatet) gjør vi koordinatene i $\vec{u}$ om til noen lettere å tolke.

In [6]:
def XYZtolatlonelev(position):
    # computes the latitude, longitude and elevation (over the ellipsoid) corresponding to position
    
    a =  6378137.0 # Earth equatorial radius
    b =  6356752.3 # Earth polar radius
    esq =  1-(b/a)**2 # squared eccentricity of earth
    
    x=position[0]
    y=position[1]
    z=position[2]

    p =  np.sqrt(x**2+y**2)
    
    coeffs =[(1-esq)*z**2, -2*(1-esq)*z**2, p**2+(1-esq)*z**2-esq**2*a**2, -2*p**2, p**2]
    kappa=np.roots(coeffs)[0].real
    
    lon = np.rad2deg(np.arctan2(y,x))
    lat= np.rad2deg(np.arctan(z*kappa/p))
    elev = np.sqrt(p**2+z**2*kappa**2)*(1/kappa-(1-esq))/esq
    
    
    return lat, lon, elev

Sett inn de to svarene du får her.

In [7]:
print(XYZtolatlonelev(u1[0:3])) # den riktige, 143moh
print(XYZtolatlonelev(u2[0:3])) # ikke bra, 19373864moh??? 

(59.65743568350251, 10.684415978898818, 143.81487057479285)
(60.176301495337384, 11.23887484393877, 19373864.229923666)


## Oppgaver

1. Hvor befinner vi oss? Tips: takk til Geir Bogfjellmo og Ola Øvstedal (NMBU) som har bidratt til øvingen!
2. Forklar hvorfor vi må ha 4 satelitter for å få et entydig svar
3. Hent noen data fro Gjøvik-området og sjekk om du får et passende svar.
4. Hva skjer hvis vi har flere enn 4 satelitter?

Vi skal prate mer om sistnevnte senere. Metoden gitt har funker fortsett, men vi må ta i bruk 'pseudo-inversen' til matrisa. Det er tett tilknyttet til minste kvadraters metode.